# TinySQL LoRA: train and evaluate the same adapter

TinySQL uses a separately trained SQL base and shares no weights with the ticket experiment. All questions are synthetic, SQL skeletons recur, and historical test/challenge splits were inspected during development. CPU is the verified execution target.

Generated datasets and weights are not committed. The default training mode generates data, trains a diagnostic base if no local base exists, and trains a new adapter. Run from a clone with the README's locked CPU environment. Colab users can upload a source-only repository ZIP; Colab execution has not been verified.


In [ ]:
import json
import pathlib
import subprocess
import sys
import zipfile
from datetime import UTC, datetime

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    archive = next(name for name in uploaded if name.endswith(".zip"))
    destination = pathlib.Path("/content/tinysql-repository")
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(destination)
    candidates = [p.parent for p in destination.rglob("uv.lock")]
    assert len(candidates) == 1, "Upload one complete repository"
    ROOT = candidates[0]
    subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.12.1"], check=True)
    subprocess.run(["uv", "python", "install", "3.12.12"], check=True)
    subprocess.run(
        [
            "uv",
            "sync",
            "--project",
            str(ROOT),
            "--locked",
            "--extra",
            "cpu",
            "--group",
            "pretraining",
            "--group",
            "demo",
        ],
        check=True,
    )
else:
    ROOT = pathlib.Path.cwd().resolve()
    while not (ROOT / "uv.lock").exists():
        assert ROOT != ROOT.parent, "Run from a repository clone"
        ROOT = ROOT.parent
TRACK = ROOT / "finetuning"


def run(*args):
    return subprocess.run(
        ["uv", "run", "--project", str(ROOT), "--no-sync", "python", *map(str, args)],
        cwd=TRACK,
        check=True,
    )


print("Repository:", ROOT)

## Choose training or externally stored historical artifacts

`RUN_TRAINING=True` creates a new adapter. If `base_small/` is absent, it also trains a new base under the run directory. Smoke mode uses 12 updates and does not reproduce historical benchmark scores. Every later cell uses `BASE_DIR`, `DATA_DIR`, and `ACTIVE_ADAPTER`.

`RUN_TRAINING=False` requires separately restored `base_small/` and `adapters/aug_r8_fast/` artifacts. There is no automatic historical-weight download or substitution.


In [ ]:
RUN_TRAINING = True
MODE = "smoke"  # 'smoke' or 'full_cpu'
PRESETS = {"smoke": 12, "full_cpu": 90}
assert MODE in PRESETS, "Choose smoke or full_cpu"
RUN_ROOT = TRACK / "runs" / ("notebook-" + datetime.now(UTC).strftime("%Y%m%d-%H%M%S-%f"))
RUN_ROOT.mkdir(parents=True, exist_ok=False)
BASE_DIR = TRACK / "base_small"
ACTIVE_ADAPTER = TRACK / "adapters/aug_r8_fast/adapter.pt"
if not RUN_TRAINING:
    assert (BASE_DIR / "model.pt").is_file() and ACTIVE_ADAPTER.is_file(), (
        "Restore the historical base and matching adapter, or set RUN_TRAINING=True. "
        "Model artifacts are not included in Git."
    )
DATA_DIR = RUN_ROOT / "data"
run(
    "-m",
    "finetuning.tinysql.data",
    "--out",
    DATA_DIR,
    "--train",
    6400,
    "--val",
    320,
    "--test",
    320,
    "--challenge",
    160,
)
if RUN_TRAINING:
    if not (BASE_DIR / "model.pt").is_file():
        BASE_DIR = RUN_ROOT / "base"
        base_options = ["--epochs", 1, "--steps", 12] if MODE == "smoke" else ["--epochs", 20]
        run(
            "-m",
            "finetuning.tinysql.train",
            "--data-dir",
            DATA_DIR,
            "--out",
            BASE_DIR,
            "--device",
            "cpu",
            "--num-threads",
            2,
            *base_options,
        )
    AUGMENTED_DATA = RUN_ROOT / "augmentation"
    run(
        "-m",
        "finetuning.lora_sql.augment",
        "--src",
        DATA_DIR / "train.jsonl",
        "--out",
        AUGMENTED_DATA / "train.jsonl",
    )
    ACTIVE_ADAPTER = RUN_ROOT / "adapter" / "adapter.pt"
    run(
        "-m",
        "finetuning.lora_sql.train_lora",
        "--base",
        BASE_DIR,
        "--data",
        AUGMENTED_DATA,
        "--out",
        ACTIVE_ADAPTER.parent,
        "--rank",
        8,
        "--alpha",
        16,
        "--lr",
        0.0003,
        "--batch",
        64,
        "--epochs",
        1,
        "--steps",
        PRESETS[MODE],
        "--seed",
        42,
        "--device",
        "cpu",
    )
print("Active base:", BASE_DIR)
print("Active adapter:", ACTIVE_ADAPTER)
print(json.loads(ACTIVE_ADAPTER.with_name("metadata.json").read_text()))

## Generate with the base and active adapter


In [ ]:
run(TRACK / "scripts/demo_compare.py", "--base", BASE_DIR, "--adapter", ACTIVE_ADAPTER)

## Evaluate the active adapter on validation only

Both models answer the same questions with greedy generation and are scored on two populated SQLite databases. A smoke subset is not a full benchmark. Historical test/challenge results remain separate.


In [ ]:
ACTIVE_REPORT = RUN_ROOT / "val.json"
run(
    "-m",
    "finetuning.lora_sql.evaluate_lora",
    "--base",
    BASE_DIR,
    "--adapter",
    ACTIVE_ADAPTER,
    "--data",
    DATA_DIR,
    "--split",
    "val",
    "--count",
    32 if MODE == "smoke" else 0,
    "--output",
    ACTIVE_REPORT,
)
print(json.loads(ACTIVE_REPORT.read_text()))

## Plot this run and its evaluation


In [ ]:
run(
    "-m",
    "finetuning.lora_sql.plots",
    "--run-dir",
    ACTIVE_ADAPTER.parent,
    "--report",
    ACTIVE_REPORT,
    "--out",
    RUN_ROOT / "plots",
)
for figure in sorted((RUN_ROOT / "plots").glob("*.png")):
    print("Generated figure:", figure)

## Regression tests


In [ ]:
run("-m", "pytest", "-q", TRACK / "tests")

## Export and download the active adapter with its exact base

The bundle contains custom inference source, base weights, tokenizer, adapter, compatibility metadata, and measured results for this adapter. Export is local; no Hub upload occurs.


In [ ]:
EXPORT = RUN_ROOT / "export"
run(
    "-m",
    "finetuning.lora_sql.publish",
    "--adapter",
    ACTIVE_ADAPTER.parent,
    "--base",
    BASE_DIR,
    "--out",
    EXPORT,
    "--report",
    ACTIVE_REPORT,
)
print((EXPORT / "README.md").read_text())
if IN_COLAB:
    import shutil

    from google.colab import files

    archive = shutil.make_archive(str(RUN_ROOT / "tinysql-export"), "zip", EXPORT)
    files.download(archive)
else:
    print("Export:", EXPORT)